In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Anish494/flyrank_ai_first_assignment"
REPO_DIR = "flyrank_ai_first_assignment"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
print(f"{len(df):,} rows ready — same filtered dataset as ML-07 baseline")

30,000 rows ready — same filtered dataset as ML-07 baseline


# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anish494/flyrank_ai_first_assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: K-Means clustering**, from the "clustering" option on this week's menu.

This fits my lane (Structured Content Archetype Clustering) directly — I'm not
predicting a known outcome (which would call for logistic regression, decision tree,
random forest, or gradient boosting), I'm discovering groups of similar pages with no
pre-existing label. K-Means is the standard, explainable starting point for this: it's
fast, its logic is easy to describe (assign each page to its nearest group center,
then update the centers), and it produces exactly the kind of hard cluster
assignment my ML-03 framing already committed to.

I will scale features first (K-Means is distance-based, so unscaled features like
impressions_90d — which ranges into the hundreds of thousands — would dominate over
CTR, which ranges 0-1). I'll pick K using both an elbow plot (inertia vs K) and a
silhouette score, matching the success metric I defined in ML-03.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_features = ["impressions_90d", "avg_position", "ctr", "word_count", "content_age_days"]
X = df[cluster_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Try a few values of K, track inertia (elbow) and silhouette score
results = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results.append({"k": k, "inertia": km.inertia_, "silhouette": sil})

results_df = pd.DataFrame(results)
print(results_df)

   k        inertia  silhouette
0  2  113602.883847    0.349145
1  3   92660.113787    0.357895
2  4   77679.053895    0.368557
3  5   61419.967092    0.383975
4  6   52913.153443    0.366932
5  7   48230.853787    0.369628
6  8   43729.038447    0.367048


**Chosen K = 5.** Silhouette score peaks at K=5 (0.384), higher than both K=4 (0.369)
and every K from 6-8 (which flatten around 0.367-0.370). Inertia's rate of improvement
also visibly slows after K=5, consistent with this being a reasonable elbow point.
Both signals agree, so I'm not choosing K=5 by only one weak piece of evidence.

## 2. Split design

Clustering has no train/test accuracy to validate in the usual sense, so instead I
check **stability**: if I re-run K-Means on two different random subsets of the same
data, do I get roughly the same cluster structure back? This is the clustering
equivalent of a grouped/held-out validation — instead of checking "does this
generalize to unseen clients," I check "is this structure real, or did it just
happen to fall out of one particular run/sample?"

In [3]:
from sklearn.model_selection import train_test_split

# Split into two random halves (not client-grouped here, since clustering is
# per-page and content items aren't nested under clients in the starter CSV
# the same way the warehouse's fact tables are)
half_a, half_b = train_test_split(df, test_size=0.5, random_state=1)

X_a = scaler.transform(half_a[cluster_features].fillna(0))
X_b = scaler.transform(half_b[cluster_features].fillna(0))

km_a = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_a)
km_b = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_b)

sil_a = silhouette_score(X_a, km_a.labels_)
sil_b = silhouette_score(X_b, km_b.labels_)
print(f"Silhouette on half A: {sil_a:.3f}")
print(f"Silhouette on half B: {sil_b:.3f}")

# Compare cluster CENTERS (in original units) to see if similar archetypes emerge
centers_a = pd.DataFrame(scaler.inverse_transform(km_a.cluster_centers_), columns=cluster_features).round(1)
centers_b = pd.DataFrame(scaler.inverse_transform(km_b.cluster_centers_), columns=cluster_features).round(1)
print("\nCluster centers, half A:")
print(centers_a.sort_values("impressions_90d", ascending=False))
print("\nCluster centers, half B:")
print(centers_b.sort_values("impressions_90d", ascending=False))

Silhouette on half A: 0.386
Silhouette on half B: 0.382

Cluster centers, half A:
   impressions_90d  avg_position   ctr  word_count  content_age_days
3         106825.8          10.5   0.3      2617.5             260.5
0           4188.8          13.7   0.3      3595.5             158.9
2           3779.4          10.5   0.4       854.4             370.2
1           1049.3          49.3   0.1      1110.0             346.6
4              5.3           5.7  35.7      1151.1             286.6

Cluster centers, half B:
   impressions_90d  avg_position   ctr  word_count  content_age_days
4         118305.2          10.7   0.3      2726.1             261.4
1           4298.7          13.7   0.3      3571.2             159.0
0           3772.2          10.5   0.4       818.3             368.0
3           1081.5          49.0   0.1      1127.0             342.1
2              3.8           7.3  45.0      1270.1             285.7


**Stability confirmed:** silhouette scores nearly match (0.386 vs 0.382), and four of
the five cluster centers line up closely across two independently sampled halves —
e.g. the largest-impression cluster sits at ~107k-118k impressions in both halves,
with matching position (~10.5) and CTR (~0.3). This is real, repeatable structure,
not an artifact of one particular sample.

**Data quality flag:** the fifth cluster in both halves shows an impossible CTR
(35.7 and 45.0 — CTR should be bounded 0-1) paired with near-zero impressions (5.3,
3.8). This is almost certainly a data artifact from a handful of rows with a
near-zero denominator producing a nonsensical ratio, not a real archetype. I will
investigate and likely filter these rows before finalizing my archetype names in the
capstone, rather than naming this cluster as if it were meaningful.

## 3. Train + compare vs my baseline

Clustering (5 unsupervised groups) and my ML-07 baseline (a single binary rule) aren't
comparable via a shared accuracy metric — they answer different questions. Instead, I
compare them the way the assignment's data actually allows: I re-apply my baseline
rule to the same rows used for clustering, then check how baseline-flagged pages are
distributed across the 5 clusters. If flagged pages cluster into just one archetype,
the baseline and my model agree cleanly. If they spread across several very different
archetypes, that's evidence the baseline's single flag is collapsing meaningfully
different situations into one bucket -- exactly the richer picture clustering is
supposed to add.

In [4]:
# Re-fit K-Means on the FULL filtered dataset (not the half-splits from Section 2)
X_full_scaled = scaler.fit_transform(df[cluster_features].fillna(0))
km_full = KMeans(n_clusters=5, random_state=42, n_init=10)
df["cluster"] = km_full.fit_predict(X_full_scaled)

# Re-apply the exact ML-07 baseline rule
df["baseline_flag"] = (
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)
)

# How are baseline-flagged pages distributed across the 5 clusters?
distribution = df.groupby("cluster")["baseline_flag"].agg(["sum", "count"])
distribution["pct_flagged"] = (distribution["sum"] / distribution["count"] * 100).round(1)
print("Baseline-flagged pages by cluster:")
print(distribution)

# Cluster centers on the full data, for reference
centers_full = pd.DataFrame(
    scaler.inverse_transform(km_full.cluster_centers_), columns=cluster_features
).round(1)
print("\nCluster centers (full data):")
print(centers_full.sort_values("impressions_90d", ascending=False))

Baseline-flagged pages by cluster:
          sum  count  pct_flagged
cluster                          
0        5995  15461         38.8
1         523   3278         16.0
2        3275  10694         30.6
3           0    158          0.0
4         168    409         41.1

Cluster centers (full data):
   impressions_90d  avg_position   ctr  word_count  content_age_days
4         112882.8          10.8   0.3      2707.4             261.1
0           4244.3          13.7   0.3      3583.4             158.9
2           3809.4          10.5   0.4       837.1             369.2
1           1062.0          49.2   0.1      1114.9             344.0
3              4.6           6.2  38.2      1174.3             288.4


**Finding:** the baseline's flag rate varies enormously by archetype — from 0% (the
data-artifact cluster, correctly excluded since these pages have almost no
impressions) up to 41.1% (the highest-traffic "champion" cluster). Most notably,
**Cluster 1** — the archetype with the objectively worst position (avg ~49, page 5+)
and worst CTR (0.1) among real archetypes — has the *lowest* real flag rate (16.0%).
This is likely because the baseline's `impressions_90d >= 500` floor filters out many
of these already-low-traffic, poorly-ranked pages before they ever reach the "declining"
check — meaning the baseline may be systematically under-flagging exactly the weakest
archetype, simply because it's also the lowest-volume one. Clustering surfaces this
gap directly; the baseline's single binary rule could not have revealed it on its own.

## 4. Errors and interpretation

For clustering, "error" doesn't mean a wrong prediction — it means a page that sits
ambiguously between two clusters, or a cluster that's internally inconsistent. I check
this by looking at each page's distance to its own cluster's center versus the
nearest other cluster's center: pages where these distances are close together are
poorly-separated, borderline cases.

In [5]:
from scipy.spatial.distance import cdist

# Distance from every point to every cluster center
distances = cdist(X_full_scaled, km_full.cluster_centers_)

# For each point: distance to its OWN cluster vs the SECOND-closest cluster
own_dist = distances[np.arange(len(df)), df["cluster"].values]
sorted_dist = np.sort(distances, axis=1)
second_closest_dist = sorted_dist[:, 1]

df["ambiguity_gap"] = second_closest_dist - own_dist  # small gap = ambiguous point

print("How many pages sit within a small margin of a second cluster (ambiguous):")
ambiguous = df[df["ambiguity_gap"] < 0.3]
print(f"{len(ambiguous):,} pages ({len(ambiguous)/len(df)*100:.1f}%) are borderline "
      f"(ambiguity_gap < 0.3)")

print("\nWhich clusters do these ambiguous pages sit in?")
print(ambiguous["cluster"].value_counts())

print("\nExample borderline pages:")
print(ambiguous[["content_id"] + cluster_features + ["cluster", "ambiguity_gap"]].sort_values("ambiguity_gap").head(5))

How many pages sit within a small margin of a second cluster (ambiguous):
2,358 pages (7.9%) are borderline (ambiguity_gap < 0.3)

Which clusters do these ambiguous pages sit in?
cluster
2    1186
0     646
1     507
4      18
3       1
Name: count, dtype: int64

Example borderline pages:
                 content_id  impressions_90d  avg_position   ctr  word_count  \
14191  content_3a75dc5dd6cc              370          30.9  0.00         NaN   
5355   content_893c17df988e              304          44.0  0.00      4043.0   
25248  content_565c92d02525             2741          22.4  0.33      1426.0   
12325  content_68a9fa6ae719               78          29.9  0.00         NaN   
777    content_3ebeade99cf0               38          50.0  0.00      4001.0   

       content_age_days  cluster  ambiguity_gap  
14191               445        2       0.000037  
5355                182        1       0.000445  
25248               223        2       0.000624  
12325               329      

**Interpretation:** 7.9% of pages (2,358) are genuinely borderline. They concentrate
almost entirely in Clusters 0, 1, and 2 (99.9% of all ambiguous cases) — the three
archetypes that all sit in the "moderate impressions, moderate-to-weak position" zone,
which are naturally closer together in feature space than the extreme, easily-separated
Cluster 4 (huge impressions) or Cluster 3 (the data-artifact cluster). This means the
model draws a clean line around the extreme cases but has real, honest uncertainty in
the middle of the distribution — which is expected and worth stating plainly rather
than pretending every page has a crisp, obvious archetype.

**Data quality issue found in this analysis:** two of the five example borderline rows
have `word_count = NaN`, which my `.fillna(0)` step silently converted to zero before
clustering. This could be artificially pulling pages with genuinely unmeasured word
counts toward the short-content cluster (Cluster 2), rather than reflecting their real
content length. Before finalizing capstone results, I should either impute this more
carefully (e.g. median word count) or exclude rows with missing word_count entirely,
rather than let a missing-data artifact quietly shape cluster assignment.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.